In [1]:
import pandas as pd
import geopandas as gpd



In [2]:
arrets_gdf = gpd.read_file("../raw/arrets-lignes.geojson")
iris_gdf = gpd.read_file("../raw/iris_raw_20260427.geojson")

In [3]:
arrets_gdf.head()

,id,route_long_name,stop_id,stop_name,stop_lon,stop_lat,operatorname,shortname,bookingrules,mode,nom_commune,code_insee,geometry
0,IDFM:C02636,Tzen4,IDFM:426760,Charles Robin,2.466885535426266,48.61321828071897,TISSE,Tzen4,NaN,Bus,Corbeil-Essonnes,91174,POINT (2.46689 48.61322)
1,IDFM:C02636,Tzen4,IDFM:3859,Lisière des Deux Parcs,2.437547444124736,48.6241385272201,TISSE,Tzen4,NaN,Bus,Évry-Courcouronnes,91228,POINT (2.43755 48.62414)
2,IDFM:C02636,Tzen4,IDFM:487114,Les Miroirs,2.428818670322126,48.63197680175785,TISSE,Tzen4,NaN,Bus,Évry-Courcouronnes,91228,POINT (2.42882 48.63198)
3,IDFM:C02636,Tzen4,IDFM:422528,Marchais Guesdon,2.4123899036159604,48.63270646476739,TISSE,Tzen4,NaN,Bus,Évry-Courcouronnes,91228,POINT (2.41239 48.63271)
4,IDFM:C02636,Tzen4,IDFM:487111,La Mare à Pilatre,2.403606236307223,48.64252152194464,TISSE,Tzen4,NaN,Bus,Ris-Orangis,91521,POINT (2.40361 48.64252)


In [4]:
iris_gdf = iris_gdf.to_crs(arrets_gdf.crs)

In [7]:
arrets_iris = gpd.sjoin(
    arrets_gdf,
    iris_gdf,
    how="inner",
    predicate="within"
)

In [8]:
arrets_iris.head()

,id_left,route_long_name,stop_id,stop_name,stop_lon,stop_lat,operatorname,shortname,bookingrules,mode,...,index_right,dep,insee_com,nom_com,iris,code_iris,nom_iris,typ_iris,geo_point_2d,id_right
0,IDFM:C02636,Tzen4,IDFM:426760,Charles Robin,2.466885535426266,48.61321828071897,TISSE,Tzen4,NaN,Bus,...,4415,91,91174,Corbeil-Essonnes,0202,911740202,Tarterêts Montagne des Glaises,H,"{'lon': 2.464351925068005, 'lat': 48.612206040...",IRIS____0000000911740202
1,IDFM:C02636,Tzen4,IDFM:3859,Lisière des Deux Parcs,2.437547444124736,48.6241385272201,TISSE,Tzen4,NaN,Bus,...,3525,91,91228,Évry-Courcouronnes,0110,912280110,Épinettes Est,H,"{'lon': 2.4427897532986265, 'lat': 48.62313344...",IRIS____0000000912280110
2,IDFM:C02636,Tzen4,IDFM:487114,Les Miroirs,2.428818670322126,48.63197680175785,TISSE,Tzen4,NaN,Bus,...,3754,91,91228,Évry-Courcouronnes,0104,912280104,Pyramides Sud,H,"{'lon': 2.429093496595049, 'lat': 48.631749433...",IRIS____0000000912280104
3,IDFM:C02636,Tzen4,IDFM:422528,Marchais Guesdon,2.4123899036159604,48.63270646476739,TISSE,Tzen4,NaN,Bus,...,3763,91,91228,Évry-Courcouronnes,0201,912280201,Canal 1,H,"{'lon': 2.4130506274844277, 'lat': 48.63318943...",IRIS____0000000912280201
4,IDFM:C02636,Tzen4,IDFM:487111,La Mare à Pilatre,2.403606236307223,48.64252152194464,TISSE,Tzen4,NaN,Bus,...,1375,91,91521,Ris-Orangis,0109,915210109,La Ferme d'Orangis,H,"{'lon': 2.4060094079107124, 'lat': 48.64271518...",IRIS____0000000915210109


In [9]:
arrets_iris.columns

Index(['id_left', 'route_long_name', 'stop_id', 'stop_name', 'stop_lon',
       'stop_lat', 'operatorname', 'shortname', 'bookingrules', 'mode',
       'nom_commune', 'code_insee', 'geometry', 'index_right', 'dep',
       'insee_com', 'nom_com', 'iris', 'code_iris', 'nom_iris', 'typ_iris',
       'geo_point_2d', 'id_right'],
      dtype='str')

In [11]:
arrets_iris = arrets_iris[["id_left","route_long_name","stop_id","stop_name","mode","code_iris","nom_iris"]]

In [12]:
arrets_iris.to_parquet("../silver/silver_arrets_iris.parquet")

In [13]:
df_silver = pd.read_parquet("../silver/silver_arrets_iris.parquet")

In [14]:
df_aggregated = (df_silver.groupby(["code_iris","nom_iris","mode"]).agg(nb_arrets=("id_left","count"),nb_lignes=("id_left","nunique")).reset_index())

In [15]:
df_aggregated.head()

,code_iris,nom_iris,mode,nb_arrets,nb_lignes
0,751010101,Saint-Germain l'Auxerrois 1,Bus,3,3
1,751010101,Saint-Germain l'Auxerrois 1,Metro,1,1
2,751010102,Saint-Germain l'Auxerrois 2,Bus,9,9
3,751010103,Saint-Germain l'Auxerrois 3,Metro,1,1
4,751010104,Saint-Germain l'Auxerrois 4,Bus,13,11


In [ ]:
df_aggregated.to_parquet("../gold/nb_arrets_lignes_iris.parquet")